# LET-7 miRNA phylogenetic analysis

Pipeline: MUSCLE (alignment) -> trimAl (format conversion) -> IQ-TREE (ML tree + ancestral reconstruction)

In [ ]:
import os
import subprocess
from pathlib import Path

WORK_DIR = Path(__file__).parent if '__file__' in dir() else Path(".")
DATA_DIR = WORK_DIR.parent / "data"
OUTPUT_DIR = WORK_DIR / "miRNA_iqtree"

INPUT_FASTA = DATA_DIR / "LET7_family_mature_ALLspecies.fa"
ALIGNED_FILE = WORK_DIR / "LET7_family_mature_ALLspecies.aln"
CLUSTAL_FILE = WORK_DIR / "LET7_family_mature_ALLspecies.clustal"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"Input: {INPUT_FASTA}")
print(f"Output dir: {OUTPUT_DIR}")
print(f"Input exists: {INPUT_FASTA.exists()}")

In [ ]:
with open(INPUT_FASTA, 'r') as f:
    lines = f.readlines()
    
n_sequences = sum(1 for line in lines if line.startswith('>'))
print(f"{n_sequences} sequences")
print()
for i, line in enumerate(lines[:20]):
    print(line.strip())

## 1. MUSCLE alignment

In [ ]:
%%time

muscle_cmd = [
    "muscle",
    "-align", str(INPUT_FASTA),
    "-output", str(ALIGNED_FILE)
]

print(f"Running: {' '.join(muscle_cmd)}")
result = subprocess.run(muscle_cmd, capture_output=True, text=True)

if result.returncode == 0:
    print(f"Done. Output: {ALIGNED_FILE}")
else:
    print(f"Error: {result.stderr}")

## 2. trimAl format conversion (FASTA -> CLUSTAL)

In [ ]:
%%time

trimal_cmd = [
    "trimal",
    "-in", str(ALIGNED_FILE),
    "-out", str(CLUSTAL_FILE),
    "-clustal"
]

print(f"Running: {' '.join(trimal_cmd)}")
result = subprocess.run(trimal_cmd, capture_output=True, text=True)

if result.returncode == 0:
    print(f"Done. Output: {CLUSTAL_FILE}")
else:
    print(f"Error: {result.stderr}")

## 3. IQ-TREE (ML tree + ancestral reconstruction)

Parameters:
- `--seed 42` / `-T AUTO` / `-m MFP` (ModelFinder Plus)
- `-B 6000` (ultrafast bootstrap) / `--ancestral` / `--sup-min 0.95`

In [ ]:
# %%time

# # Takes 10-30 minutes
# iqtree_prefix = OUTPUT_DIR / "miRNA_LET7"

# iqtree_cmd = [
#     "iqtree",
#     "-s", str(CLUSTAL_FILE),
#     "--seed", "42",
#     "-T", "AUTO",
#     "-m", "MFP",
#     "-B", "6000",
#     "--ancestral",
#     "--sup-min", "0.95",
#     "--prefix", str(iqtree_prefix)
# ]

# print(f"Running: {' '.join(iqtree_cmd)}")
# result = subprocess.run(iqtree_cmd, capture_output=True, text=True)

# if result.returncode == 0:
#     print(f"Done. Output dir: {OUTPUT_DIR}")
# else:
#     print(f"Error: {result.stderr}")

In [ ]:
# # List output files
# for f in sorted(OUTPUT_DIR.glob("*")):
#     size = f.stat().st_size / 1024
#     print(f"  {f.name:<30} ({size:.1f} KB)")

## 4. Visualization (Biopython Phylo)

In [ ]:
from Bio import Phylo
import matplotlib.pyplot as plt

tree_file = OUTPUT_DIR / "miRNA_LET7.treefile"

if tree_file.exists():
    tree = Phylo.read(str(tree_file), "newick")
    print(f"Tips: {len(tree.get_terminals())}")
    print(f"Internal nodes: {len(tree.get_nonterminals())}")
else:
    print(f"Tree file not found: {tree_file}")

In [ ]:
if tree_file.exists():
    fig, ax = plt.subplots(figsize=(20, 40))
    Phylo.draw(tree, axes=ax, do_show=False)
    ax.set_title("LET-7 miRNA Family Phylogenetic Tree", fontsize=16, fontweight='bold')
    plt.tight_layout()
    
    output_png = OUTPUT_DIR / "LET7_phylogenetic_tree.png"
    plt.savefig(output_png, dpi=150, bbox_inches='tight', facecolor='white')
    print(f"Saved: {output_png}")
    plt.show()

In [ ]:
import toytree

tree = toytree.tree("miRNA_iqtree/miRNA_LET7.treefile")

# Find internal nodes with exactly 32 leaves
for node in tree.traverse():
    if not node.is_leaf():
        leaves = node.get_leaves()
        n_leaves = len(leaves)
        if n_leaves == 32:
            print(f"Node idx: {node.idx}, leaves: {n_leaves}")
            leaf_names = [leaf.name for leaf in leaves[:5]]
            print(f"  Contains: {leaf_names}...")

print(f"\nFirst 10 tip labels: {tree.get_tip_labels()[:10]}")

In [ ]:
import toytree

tree = toytree.tree("miRNA_iqtree/miRNA_LET7.treefile")

# Find the clade containing Mmr-Let-7-P2b3_5p (20-50 leaves range)
target_leaf = "Mmr-Let-7-P2b3_5p"

for node in tree.traverse():
    if not node.is_leaf():
        leaves = node.get_leaves()
        leaf_names = [leaf.name for leaf in leaves]
        if target_leaf in leaf_names:
            n_leaves = len(leaves)
            if 20 <= n_leaves <= 50:
                print(f"Node idx: {node.idx}, leaves: {n_leaves}")
                print(f"  Contains: {leaf_names[:8]}...")
                print()

In [ ]:
import toytree

tree = toytree.tree("miRNA_iqtree/miRNA_LET7.treefile")

node_1250 = tree[1250]
leaves = node_1250.get_leaves()
leaf_names = [leaf.name for leaf in leaves]

print(f"Extracting subtree with {len(leaf_names)} leaves")

subtree = toytree.mod.extract_subtree(tree, *leaf_names)

canvas, axes, mark = subtree.draw(
    height=600,
    width=400,
    tip_labels_align=True,
    tip_labels_style={"font-size": "11px"},
    use_edge_lengths=False,
)

In [ ]:
import toytree
import toyplot.png
import toyplot.pdf
import toyplot.svg

tree = toytree.tree("miRNA_iqtree/miRNA_LET7.treefile")

node_1250 = tree[1250]
leaves = node_1250.get_leaves()
leaf_names = [leaf.name for leaf in leaves]

print(f"Extracting subtree with {len(leaf_names)} leaves")

subtree = toytree.mod.extract_subtree(tree, *leaf_names)

canvas, axes, mark = subtree.draw(
    height=600,
    width=400,
    tip_labels_align=True,
    tip_labels_style={"font-size": "11px"},
    use_edge_lengths=False,
)

toyplot.png.render(canvas, "miRNA_iqtree/P2b3_subtree.png", scale=4)
toyplot.pdf.render(canvas, "miRNA_iqtree/P2b3_subtree.pdf")
toyplot.svg.render(canvas, "miRNA_iqtree/P2b3_subtree.svg")
print("Saved to miRNA_iqtree/")

In [ ]:
import toytree
import toyplot.png
import toyplot.pdf
import toyplot.svg
from pathlib import Path

def read_fasta(fasta_path):
    """Parse FASTA file into {name: sequence} dict."""
    sequences = {}
    current_name = None
    current_seq = []
    
    with open(fasta_path, 'r') as f:
        for line in f:
            line = line.strip()
            if line.startswith('>'):
                if current_name:
                    sequences[current_name] = ''.join(current_seq)
                current_name = line[1:].split()[0]
                current_seq = []
            else:
                current_seq.append(line)
        if current_name:
            sequences[current_name] = ''.join(current_seq)
    
    return sequences

fasta_path = Path("../data/LET7_family_mature_ALLspecies.fa")
sequences = read_fasta(fasta_path)
print(f"{len(sequences)} sequences loaded")

tree = toytree.tree("miRNA_iqtree/miRNA_LET7.treefile")

node_1250 = tree[1250]
leaves = node_1250.get_leaves()
leaf_names = [leaf.name for leaf in leaves]

print(f"Extracting subtree with {len(leaf_names)} leaves")

subtree = toytree.mod.extract_subtree(tree, *leaf_names)

# Labels with sequences appended
tip_labels_with_seq = []
for name in subtree.get_tip_labels():
    seq = sequences.get(name, "N/A")
    tip_labels_with_seq.append(f"{name}  {seq}")

canvas, axes, mark = subtree.draw(
    height=700,
    width=800,
    tip_labels=tip_labels_with_seq,
    tip_labels_align=True,
    tip_labels_style={"font-size": "10px", "font-family": "monospace"},
    use_edge_lengths=False,
)

toyplot.png.render(canvas, "miRNA_iqtree/P2b3_subtree_with_seq.png", scale=4)
toyplot.pdf.render(canvas, "miRNA_iqtree/P2b3_subtree_with_seq.pdf")
print("Saved.")

In [ ]:
import json
from pathlib import Path

data_dir = Path("../data")

subtree_data = {
    "name": "P2b3_subtree",
    "node_idx": 1250,
    "n_sequences": len(leaf_names),
    "sequences": {}
}

for name in subtree.get_tip_labels():
    seq = sequences.get(name, "")
    subtree_data["sequences"][name] = seq

json_path = data_dir / "P2b3_subtree_sequences.json"
with open(json_path, 'w') as f:
    json.dump(subtree_data, f, indent=2)
print(f"JSON: {json_path}")

fasta_path = data_dir / "P2b3_subtree_sequences.fa"
with open(fasta_path, 'w') as f:
    for name in subtree.get_tip_labels():
        seq = sequences.get(name, "")
        f.write(f">{name}\n{seq}\n")
print(f"FASTA: {fasta_path}")

targets_path = data_dir / "P2b3_targets.json"
targets = {name: sequences.get(name, "") for name in subtree.get_tip_labels()}
with open(targets_path, 'w') as f:
    json.dump(targets, f, indent=2)
print(f"Targets: {targets_path}")

print(f"\n{len(subtree_data['sequences'])} sequences saved")
for i, (name, seq) in enumerate(list(subtree_data['sequences'].items())[:5]):
    print(f"  {name}: {seq}")

## 5. Ancestral state reconstruction

In [ ]:
state_file = OUTPUT_DIR / "miRNA_LET7.state"

if state_file.exists():
    with open(state_file, 'r') as f:
        for i, line in enumerate(f):
            if i >= 50:
                print("...")
                break
            print(line.rstrip())
else:
    print(f"State file not found: {state_file}")

## 6. IQ-TREE report summary

In [ ]:
iqtree_report = OUTPUT_DIR / "miRNA_LET7.iqtree"

if iqtree_report.exists():
    with open(iqtree_report, 'r') as f:
        content = f.read()
    
    for line in content.split('\n'):
        if 'Best-fit model' in line:
            print(line.strip())
        if 'Log-likelihood' in line and 'tree' in line.lower():
            print(line.strip())
        if 'Total tree length' in line:
            print(line.strip())
else:
    print(f"IQ-TREE report not found: {iqtree_report}")

## Output files

- `miRNA_LET7.treefile` - Newick phylogenetic tree
- `miRNA_LET7.iqtree` - Full analysis report
- `miRNA_LET7.state` - Ancestral state reconstruction
- `LET7_phylogenetic_tree.png` - Tree visualization